# Train the pothole detector (YOLO11n) — Colab T4

Board issue **#25** → output `best.pt` for **#58** (hand to Ado) and `pothole.onnx` for **#39**.

**Before you start**
1. `Runtime → Change runtime type → T4 GPU`.
2. Free Roboflow account → copy your API key (roboflow.com → Settings → API Keys).
3. In Colab, click the 🔑 **Secrets** panel (left bar) → add `ROBOFLOW_API_KEY` → toggle notebook access on.
   (If you skip this, cell 3 asks for the key instead. Never paste it into a cell.)
4. `Runtime → Run all`. ~25–40 min total. Everything is saved to Google Drive, so a disconnect
   doesn't lose the run — just re-run all cells and it resumes.

**Datasets** (both exported as YOLOv11, merged into one class `pothole`):
- `brad-dwyer/pothole-voxrl` v1 — 665 images, the standard baseline.
- `work-5qhak/potholes-ptvvw` v2 (`dashcam_832`) — 730 dash-cam-angle images, matches our footage.

Check each dataset's license on its Roboflow page and credit both in `pipeline/README.md`.

## 1 · GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU — Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/still-there-potholes'   # runs, weights, exports land here
import os; os.makedirs(DRIVE_DIR, exist_ok=True)
print('Saving to', DRIVE_DIR)

## 2 · Install

In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3 · Download both datasets from Roboflow

In [ ]:
import getpass
try:
    from google.colab import userdata
    RF_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    RF_KEY = None
if not RF_KEY:
    RF_KEY = getpass.getpass('Roboflow API key: ')

from roboflow import Roboflow
rf = Roboflow(api_key=RF_KEY)

DATASETS = {   # short name -> (workspace, project, version)
    'voxrl':   ('brad-dwyer', 'pothole-voxrl', 1),
    'dashcam': ('work-5qhak', 'potholes-ptvvw', 2),
}
RAW = '/content/datasets'
paths = {}
for name, (ws, proj, ver) in DATASETS.items():
    loc = f'{RAW}/{name}'
    if os.path.exists(f'{loc}/data.yaml'):
        print(name, 'already downloaded'); paths[name] = loc; continue
    v = rf.workspace(ws).project(proj).version(ver)
    try:
        v.download('yolov11', location=loc)
    except Exception as e:                       # older roboflow: same TXT format under 'yolov8'
        print('yolov11 export failed, using yolov8 format:', e)
        v.download('yolov8', location=loc)
    paths[name] = loc
paths

## 4 · One class `pothole`, then merge

Every label is rewritten to class `0 = pothole`. A class whose name contains "pothole" (any case)
is kept; any other class is dropped (printed below so you can see it). If a dataset has exactly
one class it's kept whatever it's called. Polygon labels are converted to boxes.

In [ ]:
PREP_CODE = True  # marker for the offline test
import glob, os, shutil, yaml
from collections import Counter

SPLITS = ('train', 'valid', 'test')

def class_map(names):
    names = list(names.values()) if isinstance(names, dict) else list(names)
    if len(names) == 1:
        return {0: 0}, names
    return {i: 0 for i, n in enumerate(names) if 'pothole' in str(n).lower()}, names

def fix_line(line, keep):
    parts = line.split()
    if len(parts) < 5:
        return None
    cls = int(float(parts[0]))
    if cls not in keep:
        return None
    nums = [float(x) for x in parts[1:]]
    if len(nums) > 4:                              # polygon x1 y1 x2 y2 ... -> box
        xs, ys = nums[0::2], nums[1::2]
        x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
        nums = [(x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0]
    return '0 ' + ' '.join(f'{v:.6f}' for v in nums[:4])

def find_split(root, split):
    for cand in (split, 'val' if split == 'valid' else split):
        if os.path.isdir(f'{root}/{cand}/images'):
            return f'{root}/{cand}'
    return None

def prep(name, root, merged):
    with open(f'{root}/data.yaml') as f:
        keep, names = class_map(yaml.safe_load(f)['names'])
    dropped = [n for i, n in enumerate(names) if i not in keep]
    print(f'{name}: classes {names} -> keep {[names[i] for i in keep]}' + (f', DROP {dropped}' if dropped else ''))
    if not keep:
        raise SystemExit(f'{name}: no pothole class found — check its data.yaml')
    stats = Counter()
    for split in SPLITS:
        src = find_split(root, split)
        if not src:
            continue
        for d in ('images', 'labels'):
            os.makedirs(f'{merged}/{split}/{d}', exist_ok=True)
        for img in glob.glob(f'{src}/images/*'):
            stem = os.path.splitext(os.path.basename(img))[0]
            lab = f'{src}/labels/{stem}.txt'
            lines = []
            if os.path.exists(lab):
                with open(lab) as f:
                    lines = [l for l in (fix_line(x, keep) for x in f) if l]
            new = f'{name}_{stem}'                 # prefix so file names never clash
            shutil.copy(img, f'{merged}/{split}/images/{new}{os.path.splitext(img)[1]}')
            with open(f'{merged}/{split}/labels/{new}.txt', 'w') as f:
                f.write('\n'.join(lines) + ('\n' if lines else ''))
            stats[split] += 1; stats[split + '_boxes'] += len(lines)
    print('  ', dict(stats))
    return stats

def write_yaml(path, root, has_test):
    cfg = {'path': root, 'train': 'train/images', 'val': 'valid/images', 'names': {0: 'pothole'}}
    if has_test:
        cfg['test'] = 'test/images'
    with open(path, 'w') as f:
        yaml.safe_dump(cfg, f)

def build(paths, out='/content/merged'):
    shutil.rmtree(out, ignore_errors=True)
    for name, root in paths.items():
        prep(name, root, out)
        sub = f'{out}_{name}'                      # per-dataset copy, for evaluating one angle alone
        shutil.rmtree(sub, ignore_errors=True)
        prep(name, root, sub)
        write_yaml(f'{sub}/data.yaml', sub, os.path.isdir(f'{sub}/test/images'))
    write_yaml(f'{out}/data.yaml', out, os.path.isdir(f'{out}/test/images'))
    return f'{out}/data.yaml'

if 'paths' in globals():
    MERGED_YAML = build(paths)
    print(open(MERGED_YAML).read())

## 5 · Train (≈20–30 min on a T4)

Saves every epoch to Drive. If Colab disconnects, re-run everything: this cell resumes from `last.pt`.

In [ ]:
from ultralytics import YOLO
RUN = f'{DRIVE_DIR}/runs/y11n_merged'
last, done = f'{RUN}/weights/last.pt', f'{RUN}/weights/TRAINING_DONE'

if os.path.exists(done):
    print('Already trained — skipping. Delete', done, 'to train again.')
elif os.path.exists(last):
    print('Resuming', last)
    YOLO(last).train(resume=True)
else:
    YOLO('yolo11n.pt').train(
        data=MERGED_YAML, epochs=60, imgsz=640,   # imgsz 640 = what the phone page feeds the model
        batch=-1,                                 # auto: largest batch that fits the GPU
        patience=20, cache='ram', workers=4,
        project=f'{DRIVE_DIR}/runs', name='y11n_merged', exist_ok=True,
    )
open(done, 'w').close()
BEST = f'{RUN}/weights/best.pt'
print('best weights:', BEST)

## 6 · How good is it? (dash-cam angle is the one that matters)

Rough guide: mAP50 ≥ 0.6 on the dash-cam split is good for the demo. Much lower → consider #61 (RDD2022).

In [ ]:
model = YOLO(BEST)
for name in DATASETS:
    m = model.val(data=f'/content/merged_{name}/data.yaml', split='val', imgsz=640, verbose=False)
    print(f'{name:8s}  mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}')

## 7 · Eyeball it on real images, pick the threshold

Optional: put a few **Detroit** frames (drive stills, SeeClickFix photos) in Drive at
`still-there-potholes/detroit_samples/`. Otherwise it uses dash-cam test images.
The pipeline's confidence threshold: pick the lowest value where false boxes mostly disappear.

In [ ]:
import random, matplotlib.pyplot as plt
samples = sorted(glob.glob(f'{DRIVE_DIR}/detroit_samples/*'))[:8] or \
          random.sample(glob.glob('/content/merged_dashcam/test/images/*') or glob.glob('/content/merged_dashcam/valid/images/*'), 8)
CONF = 0.30
res = model.predict(samples, conf=CONF, imgsz=640, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, r in zip(axes.flat, res):
    ax.imshow(r.plot()[..., ::-1]); ax.axis('off')
    ax.set_title(f'{len(r.boxes)} hit(s)  max conf {max([float(c) for c in r.boxes.conf] or [0]):.2f}')
plt.tight_layout(); plt.show()

## 8 · Export for the phone page (#39) and hand off (#58)

ONNX at 640, default export (no NMS baked in) — the format `apps/live/detector.js` decodes.
Both files land in Drive: share the folder link with Ado in the chat. **Never commit them to git.**

In [ ]:
onnx = YOLO(BEST).export(format='onnx', imgsz=640)
out = f'{DRIVE_DIR}/handoff'; os.makedirs(out, exist_ok=True)
shutil.copy(BEST, f'{out}/best.pt'); shutil.copy(onnx, f'{out}/pothole.onnx')
for f in sorted(os.listdir(out)):
    print(f'{f:14s} {os.path.getsize(f"{out}/{f}")/1e6:.1f} MB')
print('\nShare this Drive folder with Ado:', out)
print('Then: python3 scripts/board.py done 25   and post DONE in chat with the link (#58).')